# 同一计算：切分、check、runtime twin、Perf

固定 `f16[1,4096,2048]`，计算 `(x * x + x) * x`。
切一维被拒绝 → 切二维后 `check` → CUDA twin → `check` → 实测性能。

基准：**TileFoundry v0.0.2、NVIDIA H200**。需 CUDA 版 PyTorch、CUDA toolkit、Ninja 和 `ipykernel`。
下面折叠的是运行辅助代码；所有检查与性能数据均由本次执行生成。

In [1]:
import os, subprocess, sys, tempfile, statistics, json
from datetime import datetime, timezone
from pathlib import Path
import torch, tilefoundry

tf_root = Path(tilefoundry.__file__).resolve().parents[2]
commit = subprocess.check_output(["git", "rev-parse", "--short=7", "HEAD"], cwd=tf_root, text=True).strip()
assert commit == "4690c58", "请使用 TileFoundry v0.0.2"
_work = tempfile.TemporaryDirectory(prefix="tf-hello-")
os.chdir(_work.name)
sys.path.insert(0, _work.name)
assert torch.cuda.is_available(), "需要 CUDA GPU"
assert "H200" in torch.cuda.get_device_name(), "本例的测量基准为 H200"
os.environ.setdefault("TORCH_CUDA_ARCH_LIST", "9.0")

def cli(*args, fail=False):
    print("$ tilefoundry " + " ".join(args))
    result = subprocess.run([sys.executable, "-m", "tilefoundry.cli", *args],
                            capture_output=True, text=True, timeout=360)
    print(result.stdout + result.stderr, end="")
    assert result.returncode == (1 if fail else 0)
    return result.stdout + result.stderr

def analyze(source, fail=False):
    message = cli("analyze", source, "report.txt", "--compute-cost", "--memory", "--roofline", fail=fail)
    if fail:
        assert "needs 2097152 B in rmem" in message and "exceeds the 262144 B" in message
    else:
        report = Path("report.txt").read_text()
        print("\n".join(line for line in report.splitlines() if line.startswith("# ")))
        assert "rmem:196608" in report

def check(source, expected=False):
    args = ["check", source, "--inputs", "files:x.pt", "--device", "cuda", "--out", "output", "--fn", "equal"]
    if expected:
        args += ["--expected", "expected.pt"]
    assert "mismatched 0" in cli(*args)

print(f"TileFoundry v0.0.2 ({commit}) · PyTorch {torch.__version__} · {torch.cuda.get_device_name()}")
print(f"CUDA runtime: {torch.version.cuda} · GPU index: {os.environ.get('CUDA_VISIBLE_DEVICES', 'default')}")
print("UTC:", datetime.now(timezone.utc).isoformat(timespec="seconds"))

def measure(fn):
    for _ in range(10):
        fn(x)
    samples = []
    for _ in range(10):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(100):
            fn(x)
        end.record()
        end.synchronize()
        samples.append(start.elapsed_time(end) * 1000 / 100)
    return {"median_us": statistics.median(samples), "min_us": min(samples),
            "max_us": max(samples), "samples_us": samples}


TileFoundry v0.0.2 (4690c58) · PyTorch 2.13.0+cu130 · NVIDIA H200
CUDA runtime: 13.0 · GPU index: 0
UTC: 2026-09-07T10:57:25+00:00


## 1. PyTorch reference

每一步保留 fp16 舍入。保存同一输入与参考输出，后续使用显式的 `equal` 判据。

In [2]:
def reference(x):  # x: float16[1, 4096, 2048]
    a = x * x
    b = a + x
    return b * x

torch.manual_seed(0)
x = (torch.rand(1, 4096, 2048) * 2 - 1).half().cuda()
torch.save(x.cpu(), "x.pt")
torch.save(reference(x).cpu(), "expected.pt")

## 2. 第一刀：沿行切 8 份

每份输入为 `512 × 2048 × 2 B = 2 MiB`，超过 target 声明的 256 KiB rmem 容量。

In [3]:
%%writefile hello_placed_toobig.py
from __future__ import annotations

from tilefoundry import func, module
from tilefoundry.dsl import Mesh, Tensor, tf
from tilefoundry.ir.types.shard import Topology
from tilefoundry.target import CudaTarget

@module(entry="chain", target=CudaTarget("nvidia.h200_sxm"),
        topologies=(Topology("cta", 8),))
class Placed:
    @func
    def chain(x: Tensor[(1, 4096, 2048), "f16"]) -> Tensor[(1, 4096, 2048), "f16"]:
        with Mesh(("cta",), layout=(8,), names=("tile",)) as cta:
            xr = tf.reshard(x, (1, 4096 @ cta.tile, 2048), "rmem")
            a = tf.mul(xr, xr)
            b = tf.add(a, xr)
            return tf.reshard(tf.mul(b, xr), (1, 4096 @ cta.tile, 2048), "gmem")

Writing hello_placed_toobig.py


In [4]:
analyze("hello_placed_toobig.py:Placed.chain", fail=True)

$ tilefoundry analyze hello_placed_toobig.py:Placed.chain report.txt --compute-cost --memory --roofline


tilefoundry: error: function 'chain': value 'v0:14' needs 2097152 B in rmem, which exceeds the 262144 B the target states for that level


## 3. 第二刀：再沿列切 32 份，然后 check

形成 `8×32` 个 CTA 分片，每份输入 64 KiB。CTA 内用 `32×8` 个线程，最内侧连续 8 个 fp16 留给同一线程，以便一次访问 16 字节。
HIR 的布局展开为：行 `8×16×32=4096`，列 `32×8×8=2048`。末尾未切分的 `8` 表达连续数据组，向量访存指令由 CUDA 实现。
分析通过后，用同一输入对照 PyTorch reference。

In [5]:
%%writefile hello_two_cuts.py
from tilefoundry import func, module
from tilefoundry.dsl import Mesh, Tensor, Topology, tf
from tilefoundry.target import CudaTarget

@module(entry="chain", target=CudaTarget("nvidia.h200_sxm"),
        topologies=(Topology("cta", 256), Topology("thread", 256)))
class Placed:
    @func
    def chain(x: Tensor[(1, 4096, 2048), "f16"]):
        with Mesh(("cta",), layout=(8, 32), names=("row", "col")) as cta:
            with Mesh(("thread",), layout=(32, 8), names=("row", "col")) as thread:
                # 行：8 个 CTA × 16 轮 × 32 个线程；列：32 个 CTA × 8 个线程 × 连续 8 个值。
                xr = tf.reshard(x, (1, 8 @ cta.row, 16, 32 @ thread.row,
                                    32 @ cta.col, 8 @ thread.col, 8), "rmem")
                a = tf.mul(xr, xr)
                b = tf.add(a, xr)
                return tf.reshard(tf.mul(b, xr),
                                 (1, 8 @ cta.row, 16, 32 @ thread.row,
                                     32 @ cta.col, 8 @ thread.col, 8), "gmem")

Writing hello_two_cuts.py


In [6]:
analyze("hello_two_cuts.py:Placed.chain")
check("hello_two_cuts.py:Placed.chain", expected=True)

$ tilefoundry analyze hello_two_cuts.py:Placed.chain report.txt --compute-cost --memory --roofline


# analysis target=nvidia.h200_sxm module=Placed function=chain topology=cta
# selection requested=compute-cost,memory,roofline executed=compute-cost,memory,roofline
# compute-cost flops=f16:25165824@98304
# traffic traffic=gmem:r16777216/w16777216@r65536/w65536,rmem:r117440512/w67108864@r458752/w262144
# peak-footprint=gmem:16842752,rmem:196608
# roofline ideal-ns=6991 bound-by=memory
$ tilefoundry check hello_two_cuts.py:Placed.chain --inputs files:x.pt --device cuda --out output --fn equal --expected expected.pt


hello_two_cuts.py:Placed.chain
  reference: expected.pt
  inputs:    files:x.pt; activations actual f16 (declared none); files x.pt: 1 tensor(s) f16[1, 4096, 2048]

  output   f16[1,4096,2048]   ref_norm 1696.23
    equal                              mismatched 0 elements 8.38861e+06 PASS

PASS


## 4. 向量化 CUDA custom op → twin → check → Perf

`grid=(32,8)` 对应 CTA 的列、行切分；`block=(8,32)` 对应线程的列、行分工。
`uint4` 一次读写 16 字节（8 个 fp16），`Half8` 在打包数据和 `half2` 之间转换，内部用 4 组 `half2` 完成计算。`--fmad=false` 保留分步运算。
`@runtime_module(Placed)` 将这份实现关联到同步修改后的 HIR。

In [7]:
%%writefile hello_twin_cuda.py
import torch
from torch.utils.cpp_extension import load_inline
from tilefoundry.runtime import runtime_func, runtime_module
from hello_two_cuts import Placed


load_inline(
    name="tf_blog_hello_cuda_inline",
    cpp_sources=r"""#include <ATen/ATen.h>
#include <torch/library.h>
at::Tensor chain_cuda(const at::Tensor& x);
TORCH_LIBRARY(tf_blog_hello, m) { m.def("chain(Tensor x) -> Tensor"); }
TORCH_LIBRARY_IMPL(tf_blog_hello, CUDA, m) { m.impl("chain", &chain_cuda); }
""",
    cuda_sources=r"""#include <ATen/ATen.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAGuard.h>
#include <c10/cuda/CUDAException.h>
#include <cuda_fp16.h>
#include <cstdint>

union Half8 { uint4 packed; half2 pairs[4]; };

__global__ void chain(const half* __restrict__ input, half* __restrict__ output) {
    int c = threadIdx.x * 8;
    for (int r = threadIdx.y; r < 512; r += blockDim.y) {
        int i = (blockIdx.y * 512 + r) * 2048 + blockIdx.x * 64 + c;
        Half8 values{*reinterpret_cast<const uint4*>(input + i)};
        #pragma unroll
        for (int j = 0; j < 4; ++j) {
            half2 x = values.pairs[j];
            half2 a = __hmul2(x, x);
            half2 b = __hadd2(a, x);
            values.pairs[j] = __hmul2(b, x);
        }
        *reinterpret_cast<uint4*>(output + i) = values.packed;
    }
}

at::Tensor chain_cuda(const at::Tensor& x) {
    TORCH_CHECK(x.is_cuda() && x.is_contiguous() && x.scalar_type() == at::kHalf,
                "expected a contiguous CUDA float16 tensor");
    TORCH_CHECK(x.sizes() == at::IntArrayRef({1, 4096, 2048}), "expected [1,4096,2048]");
    TORCH_CHECK(reinterpret_cast<std::uintptr_t>(x.data_ptr()) % 16 == 0,
                "expected 16-byte aligned input");
    c10::cuda::CUDAGuard guard(x.device());
    auto y = at::empty_like(x);
    auto stream = at::cuda::getCurrentCUDAStream();
    chain<<<dim3(32, 8), dim3(8, 32), 0, stream>>>(
        reinterpret_cast<const half*>(x.data_ptr<at::Half>()),
        reinterpret_cast<half*>(y.data_ptr<at::Half>()));
    C10_CUDA_KERNEL_LAUNCH_CHECK();
    return y;
}
""",
    is_python_module=False,
    no_implicit_headers=True,
    extra_cflags=["-O3"],
    extra_cuda_cflags=["-O3", "--fmad=false"],
)


@runtime_module(Placed)
class InlineTwin:
    @runtime_func
    def chain(self, x):
        return torch.ops.tf_blog_hello.chain(x)

Writing hello_twin_cuda.py


不传 `--expected`，这次参考自动变为 `Placed.chain` 的 HIR evaluator。`equal` 比较同一输入的输出。

In [8]:
check("hello_twin_cuda.py:InlineTwin")

$ tilefoundry check hello_twin_cuda.py:InlineTwin --inputs files:x.pt --device cuda --out output --fn equal


hello_twin_cuda.py:InlineTwin
  reference: evaluator on Placed.chain
  inputs:    files:x.pt; activations actual f16 (declared none); files x.pt: 1 tensor(s) f16[1, 4096, 2048]

  output   f16[1,4096,2048]   ref_norm 1696.23
    equal                              mismatched 0 elements 8.38861e+06 PASS

PASS


检查通过后，测量同一 GPU 上的 PyTorch reference 与 twin。编译不计时；各预热 10 次，再测 10 组，每组 100 次，报告 CUDA event 平均单次耗时的中位数及范围。包含调用期间的分配与提交间隔，不是孤立的 kernel 指令时间。

In [9]:
from hello_twin_cuda import InlineTwin

runtime = InlineTwin()
perf = {"PyTorch reference": measure(reference), "CUDA twin": measure(runtime.chain)}
for name, result in perf.items():
    print(f"{name}: {result['median_us']:.2f} μs  "
          f"(范围 {result['min_us']:.2f}–{result['max_us']:.2f} μs)")
print(f"reference / twin: {perf['PyTorch reference']['median_us'] / perf['CUDA twin']['median_us']:.2f}×")

PyTorch reference: 34.67 μs  (范围 34.56–34.74 μs)
CUDA twin: 7.68 μs  (范围 7.65–7.80 μs)
reference / twin: 4.51×
